<a href="https://colab.research.google.com/github/Dharshanthisside/ITA0616---Machine-learning/blob/main/Exp_2Candidate_Elimination_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Create the CSV file inside Colab (skip if already created in program 1)
csv_text = """Sky,AirTemp,Humidity,Wind,Water,Forecast,EnjoySport
Sunny,Warm,Normal,Strong,Warm,Same,Yes
Sunny,Warm,High,Strong,Warm,Same,Yes
Rainy,Cold,High,Strong,Warm,Change,No
Sunny,Warm,High,Strong,Cool,Change,Yes
"""
with open('enjoysport.csv', 'w') as f:
    f.write(csv_text)

data = pd.read_csv('enjoysport.csv')
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values
n = X.shape[1]

domains = [sorted(set(X[:, j])) for j in range(n)]   # possible values per attribute

def covers(h, x):
    return all(hj == '?' or hj == xj for hj, xj in zip(h, x))

def more_general(h1, h2):
    """True if h1 is more general than or equal to h2."""
    return all(a == '?' or a == b for a, b in zip(h1, h2))

# Initialise boundaries
S = None
G = [['?'] * n]

for i, (x, label) in enumerate(zip(X, y), 1):
    if label == 'Yes':
        G = [g for g in G if covers(g, x)]
        S = list(x) if S is None else [s if s == xj else '?' for s, xj in zip(S, x)]
    else:
        new_G = []
        for g in G:
            if not covers(g, x):
                new_G.append(g)
                continue
            for j in range(n):
                if g[j] == '?':
                    for v in domains[j]:
                        if v != x[j] and S is not None and S[j] == v:
                            h = g.copy()
                            h[j] = v
                            new_G.append(h)
        # keep only maximally general members, remove duplicates
        unique = []
        for h in new_G:
            if h not in unique:
                unique.append(h)
        G = [h for h in unique
             if not any(h != o and more_general(o, h) for o in unique)]
    print(f"Step {i}:\n  S = {S}\n  G = {G}")

print("\nFinal Specific boundary S:", S)
print("Final General boundary G:", G)

Step 1:
  S = ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
  G = [['?', '?', '?', '?', '?', '?']]
Step 2:
  S = ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
  G = [['?', '?', '?', '?', '?', '?']]
Step 3:
  S = ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
  G = [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', 'Same']]
Step 4:
  S = ['Sunny', 'Warm', '?', 'Strong', '?', '?']
  G = [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?']]

Final Specific boundary S: ['Sunny', 'Warm', '?', 'Strong', '?', '?']
Final General boundary G: [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?']]
